# EZNX-ATLAS-A — Scientific Reports: Complete Training Campaign (Google Colab)

**170 runs / 150 unique GPU runs** covering primary ablation (3 variants × 20 seeds),
architecture sensitivity (GLU-width), loss-weight sensitivity (LAUC),
and data-augmentation sensitivity.

## Execution order — run cells in sequence: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → ...

| Cell | What it does | Notes |
|------|-------------|-------|
| 1 | GPU / env check | Expect T4 (Colab Free) |
| 2 | Install wfdb 4.3.0 + pyarrow | Skips if already installed |
| 3 | Mount Drive + path config | Opens Drive auth once |
| 4 | Bootstrap code from GitHub | Clones training scripts (~30 s) |
| 5 | Download PTB-XL from PhysioNet | ~1.7 GB records100, ~10-20 min |
| 6 | Restore JSON results from Drive | Enables auto-resume |
| 7 | Build index_complete.parquet | Skips if parquet exists |
| 8 | Dry run — print 170-run plan | Optional verification step |
| 9 | **Group A** (60 runs, ~10 h) + Drive backup | 3 variants × 20 seeds |
| 10 | **Group B** (60 runs, ~7 h) + Drive backup | Architecture sensitivity |
| 11 | **Group C** (40 runs, ~7 h) + Drive backup | LAUC / aug sensitivity |
| 12 | **Group D** (10 runs, ~2 h) + Drive backup | Remaining ablations |
| 13 | Post-training evaluation (4 scripts) | Stats, calibration, subgroups, missingness |
| 14 | Package results + save to Drive | Final zip for download |

## Session planning (Colab Free — T4, ~12 h limit)

| Session | Run | Approx. time |
|---------|-----|-------------|
| 1 | Setup (Cells 1-8) + Group A (Cell 9) | ~11 h |
| 2 | Setup (Cells 1-7) + Group B (Cell 10) | ~8 h |
| 3 | Setup (Cells 1-7) + Groups C+D (Cells 11-12) + Evaluation (Cell 13) | ~10 h |

## Quick resume after session timeout

1. Re-run **Cells 1 → 7** (~15-20 min, dominated by PTB-XL re-download in Cell 5)
2. **Cell 6** (Restore from Drive) automatically retrieves completed JSON results
3. Resume at the appropriate **Group cell** (9 / 10 / 11 / 12)

> **Seeds**: 2024-2043 (20 consecutive, year of study initiation — no cherry-picking)  
> **Idempotent**: any completed run is detected by its `results_{run_name}.json` and skipped.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — GPU / Environment Verification                               ║
# ║  Expected: NVIDIA T4 (Colab Free) with CUDA 11.x / 12.x               ║
# ║  If you see a CPU runtime, go to Runtime > Change runtime type > GPU   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys

print('=== GPU ===')
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
print(r.stdout.strip() or 'nvidia-smi not found — no GPU attached!')

import torch
print(f'\nPyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU avail: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f'GPU name : {dev.name}')
    print(f'VRAM     : {dev.total_memory // 1024**2:,} MB')
    if 'T4' not in dev.name:
        print(f'  NOTE: Expected T4 on Colab Free; got {dev.name}.')
        print('  Training times may differ from estimates in the header.')
else:
    raise RuntimeError(
        'No GPU detected. Go to Runtime > Change runtime type > Hardware accelerator > GPU.'
    )

import numpy as np, sklearn, scipy
print(f'\nNumPy    : {np.__version__}')
print(f'sklearn  : {sklearn.__version__}')
print(f'scipy    : {scipy.__version__}')
print(f'Python   : {sys.version.split()[0]}')
print('=== OK ===')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Install dependencies                                         ║
# ║  wfdb and pyarrow are not pre-installed on Colab Python images.        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, importlib


def _install(pkg_import, pip_spec, required_ver=None):
    try:
        mod = importlib.import_module(pkg_import)
        ver = getattr(mod, '__version__', '?')
        if required_ver and ver != required_ver:
            print(f'  Re-installing {pkg_import} {ver} -> {required_ver} ...')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_spec], check=True)
            importlib.reload(mod)
        else:
            print(f'  {pkg_import} {ver}  (ok)')
    except ImportError:
        print(f'  Installing {pip_spec} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_spec], check=True)


_install('wfdb',    'wfdb==4.3.0',   required_ver='4.3.0')
_install('pyarrow', 'pyarrow>=14.0')

import wfdb, pyarrow
print(f'\nwfdb     : {wfdb.__version__}')
print(f'pyarrow  : {pyarrow.__version__}')
print('Dependencies ready.')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Mount Google Drive + path configuration                      ║
# ║                                                                         ║
# ║  Drive layout (persistent across sessions):                            ║
# ║    MyDrive/EZNX_ATLAS_A/                                               ║
# ║      json_results/   <-- results_*.json backups (~5 KB × 170 = <1 MB) ║
# ║      npz_archive/    <-- zipped probability arrays (~1.2 GB total)     ║
# ║      analysis_results/ <-- LaTeX tables, stats JSON                    ║
# ║                                                                         ║
# ║  Local paths (ephemeral — lost on session end):                        ║
# ║    /content/ptb-xl-1.0.3/  PTB-XL data (re-downloaded each session)   ║
# ║    /content/runs/           checkpoints + results JSON                 ║
# ║    /content/results/        analysis outputs                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from google.colab import drive
from pathlib import Path
import os

# Mounts Drive; opens browser auth popup on first call per session.
drive.mount('/content/drive')

# ── Local (ephemeral) paths ─────────────────────────────────────────────────
WORKING      = Path('/content')
PTB_ROOT     = WORKING / 'ptb-xl-1.0.3'
INDEX_PATH   = WORKING / 'index_complete.parquet'
RUNS_DIR     = WORKING / 'runs'
RESULTS_DIR  = WORKING / 'results'

for _d in [PTB_ROOT, RUNS_DIR, RESULTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Drive (persistent) paths ────────────────────────────────────────────────
DRIVE_BASE    = Path('/content/drive/MyDrive/EZNX_ATLAS_A')
DRIVE_JSON    = DRIVE_BASE / 'json_results'
DRIVE_NPZ     = DRIVE_BASE / 'npz_archive'
DRIVE_RESULTS = DRIVE_BASE / 'analysis_results'

for _d in [DRIVE_JSON, DRIVE_NPZ, DRIVE_RESULTS]:
    _d.mkdir(parents=True, exist_ok=True)

print('=== Local paths ===')
print(f'  PTB_ROOT    : {PTB_ROOT}')
print(f'  INDEX_PATH  : {INDEX_PATH}')
print(f'  RUNS_DIR    : {RUNS_DIR}')
print(f'  RESULTS_DIR : {RESULTS_DIR}')
print('=== Drive paths ===')
print(f'  DRIVE_BASE  : {DRIVE_BASE}')
print(f'  DRIVE_JSON  : {DRIVE_JSON}')
print(f'  DRIVE_NPZ   : {DRIVE_NPZ}')

# Environment variables inherited by subprocess calls
os.environ['EZNX_DATA_REAL']  = str(PTB_ROOT)
os.environ['EZNX_INDEX_PATH'] = str(INDEX_PATH)
os.environ['EZNX_RUNS_DIR']   = str(RUNS_DIR)
print('\nEnvironment variables set.')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Bootstrap training code from GitHub                          ║
# ║                                                                         ║
# ║  Clones the public repo and locates the kaggle_train/ subdirectory.    ║
# ║  Edit REPO_URL / REPO_REF below if needed.                             ║
# ║  WARNING: repo must be PUBLIC and contain kaggle_train/ before launch. ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os, shutil, subprocess
from pathlib import Path

WORKING = Path('/content')

REPO_URL    = os.environ.get('EZNX_REPO_URL',
              'https://github.com/ezynsegnane/ezyx-atlas-a_github_code.git')
REPO_REF    = os.environ.get('EZNX_REPO_REF', '')      # empty = default branch
REPO_SUBDIR = os.environ.get('EZNX_REPO_SUBDIR', 'kaggle_train')
CLONE_DIR   = WORKING / 'repo_src'

REQUIRED_SCRIPTS = [
    'atlas_a_v5_multiseed_v2.py',
    'run_experiments_v2.py',
    'compute_calibration.py',
    'compute_subgroups.py',
    'analyze_multiseed_v2.py',
    'evaluate_missingness_v2.py',
    'eznx_loader_v2.py',
    'eznx_model_v5.py',
    'index_construction.py',
]

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)

clone_cmd = ['git', 'clone', '--depth', '1']
if REPO_REF:
    clone_cmd += ['--branch', REPO_REF]
clone_cmd += [REPO_URL, str(CLONE_DIR)]

print(f'Cloning: {REPO_URL}')
print(f'Branch : {REPO_REF or "<default>"}')
subprocess.run(clone_cmd, check=True)

CODE_DIR = None
for candidate in [CLONE_DIR / REPO_SUBDIR, CLONE_DIR]:
    if candidate.exists() and all((candidate / s).exists() for s in REQUIRED_SCRIPTS):
        CODE_DIR = candidate
        break

if CODE_DIR is None:
    raise FileNotFoundError(
        'Training scripts not found in the cloned repo.\n'
        f'Checked: {CLONE_DIR / REPO_SUBDIR} and {CLONE_DIR}\n'
        'Make sure kaggle_train/ is committed and pushed to GitHub.'
    )

print(f'\nCODE_DIR = {CODE_DIR}')
for s in REQUIRED_SCRIPTS:
    kb = (CODE_DIR / s).stat().st_size // 1024
    print(f'  {s:<45} {kb:>4} KB')

os.environ['EZNX_CODE_DIR'] = str(CODE_DIR)
print('\nEZNX_CODE_DIR set.')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Download PTB-XL 1.0.3 from PhysioNet                        ║
# ║                                                                         ║
# ║  Selective download: ptbxl_database.csv + scp_statements.csv           ║
# ║                      + records100/ only  (~1.7 GB)                     ║
# ║  records500 is NOT downloaded (saves ~8 GB; model trains at 100 Hz).  ║
# ║                                                                         ║
# ║  /content/ is ephemeral — this cell re-runs every session (~10-20 min) ║
# ║  but skips automatically if files are already present.                 ║
# ║                                                                         ║
# ║  Free PhysioNet account: https://physionet.org/register/               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, getpass, shutil, os
from pathlib import Path

PTB_ROOT = Path('/content/ptb-xl-1.0.3')
PTB_ROOT.mkdir(exist_ok=True)

_n_dat = len(list(PTB_ROOT.rglob('*.dat')))
_have  = (
    (PTB_ROOT / 'ptbxl_database.csv').exists()
    and (PTB_ROOT / 'scp_statements.csv').exists()
    and (PTB_ROOT / 'records100').exists()
    and _n_dat >= 21000
)

if _have:
    print(f'PTB-XL already present ({_n_dat:,} .dat files) — skipping download.')
else:
    print('Enter PhysioNet credentials (free account at physionet.org/register)')
    _user = input('PhysioNet username: ').strip()
    _pass = getpass.getpass('PhysioNet password (hidden): ')

    BASE_URL = 'https://physionet.org/files/ptb-xl/1.0.3'

    # Write credentials to .netrc so they are not exposed in process arguments
    _netrc = Path('/root/.netrc')
    _netrc.write_text(
        f'machine physionet.org\n'
        f'login {_user}\n'
        f'password {_pass}\n',
        encoding='utf-8'
    )
    _netrc.chmod(0o600)
    del _user, _pass   # do not keep in namespace
    print('Credentials stored in /root/.netrc (chmod 600).')

    # ── Step 1: CSV metadata files (~2 MB) ─────────────────────────────────
    for _fname in ['ptbxl_database.csv', 'scp_statements.csv']:
        _dst = PTB_ROOT / _fname
        if not _dst.exists():
            print(f'Downloading {_fname} ...')
            _r = subprocess.run(
                ['wget', '-q', '-c', '--netrc',
                 '-O', str(_dst), f'{BASE_URL}/{_fname}'],
                capture_output=True, text=True
            )
            if _r.returncode != 0 or not _dst.exists():
                print('STDERR:', _r.stderr[:500])
                raise RuntimeError(
                    f'Failed to download {_fname}.\n'
                    'Check that your PhysioNet credentials are correct and '
                    'that you have completed the required Data Use Agreement.'
                )
        else:
            print(f'{_fname} already present — skip.')

    # ── Step 2: records100/ (~1.7 GB, recursive) ───────────────────────────
    _tmp = Path('/content/_ptbxl_dl_tmp')
    _tmp.mkdir(exist_ok=True)

    print(f'\nDownloading records100/ (~1.7 GB) — expected 10-20 min on Colab ...')
    # stdout/stderr not captured so the download progress is visible in the cell
    _r2 = subprocess.run([
        'wget', '-r', '-N', '-c', '-np', '--netrc',
        '--reject=index.html*',
        '-P', str(_tmp),
        f'{BASE_URL}/records100/',
    ])
    if _r2.returncode != 0:
        raise RuntimeError(
            'records100 download exited with non-zero code.\n'
            'Re-run this cell to resume (-c flag continues partial downloads).'
        )

    # wget creates mirror tree: _tmp/physionet.org/files/ptb-xl/1.0.3/records100/
    _mirror = _tmp / 'physionet.org' / 'files' / 'ptb-xl' / '1.0.3' / 'records100'
    if not _mirror.exists():
        _found = list(_tmp.rglob('records100'))
        if _found:
            _mirror = _found[0]
        else:
            raise FileNotFoundError(
                f'records100/ not found under {_tmp} after wget.\n'
                'The download may be incomplete. Re-run this cell.'
            )

    _dst_rec = PTB_ROOT / 'records100'
    if _dst_rec.exists():
        shutil.rmtree(_dst_rec)
    shutil.move(str(_mirror), str(PTB_ROOT))
    shutil.rmtree(_tmp, ignore_errors=True)

    # Clean up .netrc
    Path('/root/.netrc').unlink(missing_ok=True)
    print('\nDownload complete. .netrc removed.')

# ── Verification ────────────────────────────────────────────────────────────
print('\n=== Verification ===')
for _p in [PTB_ROOT / 'ptbxl_database.csv', PTB_ROOT / 'scp_statements.csv']:
    assert _p.exists(), f'MISSING: {_p}'
    print(f'  OK  {_p.name}  ({_p.stat().st_size // 1024:,} KB)')
_n_dat = len(list(PTB_ROOT.rglob('*.dat')))
_n_hea = len(list(PTB_ROOT.rglob('*.hea')))
print(f'  OK  records100/: {_n_dat:,} .dat  {_n_hea:,} .hea')
if _n_dat < 21000:
    print(f'  WARNING: expected ~21799 .dat files; found {_n_dat}. Re-run to resume wget -c.')
print(f'\nPTB_ROOT = {PTB_ROOT}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Restore JSON results from Drive (auto-resume)                ║
# ║                                                                         ║
# ║  Copies results_*.json from Drive → RUNS_DIR so the orchestrator's    ║
# ║  idempotency check (skip if JSON exists) works after a restart.        ║
# ║                                                                         ║
# ║  Safe to re-run: only copies missing files.                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import shutil
from pathlib import Path

RUNS_DIR   = Path('/content/runs')
DRIVE_JSON = Path('/content/drive/MyDrive/EZNX_ATLAS_A/json_results')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if not DRIVE_JSON.exists():
    print('No Drive backup found — starting fresh (first session).')
    print('After the first group completes, results will be backed up automatically.')
else:
    _all_drive = sorted(DRIVE_JSON.rglob('results_*.json'))
    _restored  = 0
    _skipped   = 0
    for _src in _all_drive:
        _rel = _src.relative_to(DRIVE_JSON)
        _dst = RUNS_DIR / _rel
        _dst.parent.mkdir(parents=True, exist_ok=True)
        if not _dst.exists():
            shutil.copy2(_src, _dst)
            _restored += 1
        else:
            _skipped += 1
    print(f'Drive -> local: {len(_all_drive)} JSON files on Drive')
    print(f'  Restored : {_restored}  (new files copied locally)')
    print(f'  Skipped  : {_skipped}   (already present)')

_n_local = len(list(RUNS_DIR.rglob('results_*.json')))
print(f'\nRUNS_DIR: {_n_local}/170 runs already completed (will be auto-skipped).')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Build index_complete.parquet                                 ║
# ║                                                                         ║
# ║  Two-step pipeline:                                                    ║
# ║    Step 1: engineer metadata features → index_mm_core.parquet          ║
# ║    Step 2: merge scp_codes + filename columns → index_complete.parquet ║
# ║                                                                         ║
# ║  Idempotent: skips rebuild if index_complete.parquet already exists.   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os
from pathlib import Path

WORKING    = Path('/content')
PTB_ROOT   = WORKING / 'ptb-xl-1.0.3'
INDEX_PATH = WORKING / 'index_complete.parquet'
CODE_DIR   = Path(os.environ['EZNX_CODE_DIR'])

if INDEX_PATH.exists():
    print('index_complete.parquet already exists — skipping rebuild.')
else:
    print('Building index (Step 1: features, Step 2: merge labels) ...')
    _r = subprocess.run([
        sys.executable, str(CODE_DIR / 'index_construction.py'),
        '--data-root', str(PTB_ROOT),
        '--out-dir',   str(WORKING),
    ], capture_output=True, text=True)
    print(_r.stdout[-4000:] if _r.stdout else '(no stdout)')
    if _r.returncode != 0:
        print('STDERR:', _r.stderr[-2000:])
        raise RuntimeError('index_construction.py failed — see STDERR above.')

import pandas as pd
_idx = pd.read_parquet(INDEX_PATH)
print(f'\nindex_complete.parquet: {len(_idx):,} rows, {len(_idx.columns)} columns')
print('Fold distribution:')
print(_idx.groupby('strat_fold').size().rename('count').to_string())

_req_cols = [
    'ecg_id', 'patient_id', 'strat_fold', 'scp_codes',
    'filename_lr', 'filename_hr', 'age_z', 'sex01', 'meta_present_strict',
]
_missing = [c for c in _req_cols if c not in _idx.columns]
assert not _missing, f'CRITICAL: missing columns: {_missing}'
assert len(_idx) == 21799, f'Expected 21799 rows, got {len(_idx)}'
print(f'\nAll integrity checks passed ({len(_idx):,} records, {len(_idx.columns)} columns).')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Dry run: print the full 170-experiment plan (optional)       ║
# ║  Useful to verify run names and group composition before launching.    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os
from pathlib import Path

WORKING    = Path('/content')
PTB_ROOT   = WORKING / 'ptb-xl-1.0.3'
INDEX_PATH = WORKING / 'index_complete.parquet'
RUNS_DIR   = WORKING / 'runs'
CODE_DIR   = Path(os.environ['EZNX_CODE_DIR'])

_r = subprocess.run([
    sys.executable, str(CODE_DIR / 'run_experiments_v2.py'),
    '--data_root',  str(PTB_ROOT),
    '--index_path', str(INDEX_PATH),
    '--runs_dir',   str(RUNS_DIR),
    '--dry_run',
], capture_output=True, text=True)
print(_r.stdout)
if _r.returncode != 0:
    print('STDERR:', _r.stderr[-1000:])


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Group A: 3 variants × 20 seeds = 60 runs                    ║
# ║                                                                         ║
# ║  Variants: none (ECG-only) | demo | demo+anthro                        ║
# ║  Seeds: 2024-2043                                                      ║
# ║  Estimated wall time on T4: ~10 min/run × 60 = ~10 h                  ║
# ║  Already-completed runs are skipped automatically.                     ║
# ║                                                                         ║
# ║  After completion: JSON results are backed up to Drive.                ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, shutil, os
from pathlib import Path

WORKING     = Path('/content')
PTB_ROOT    = WORKING / 'ptb-xl-1.0.3'
INDEX_PATH  = WORKING / 'index_complete.parquet'
RUNS_DIR    = WORKING / 'runs'
RESULTS_DIR = WORKING / 'results'
CODE_DIR    = Path(os.environ['EZNX_CODE_DIR'])
DRIVE_JSON  = Path('/content/drive/MyDrive/EZNX_ATLAS_A/json_results')
RESULTS_DIR.mkdir(exist_ok=True)

print('=' * 70)
print('GROUP A — ECG-only / Demo / Demo+Anthro  (3 variants x 20 seeds)')
print('=' * 70)

_r = subprocess.run([
    sys.executable, str(CODE_DIR / 'run_experiments_v2.py'),
    '--data_root',  str(PTB_ROOT),
    '--index_path', str(INDEX_PATH),
    '--runs_dir',   str(RUNS_DIR),
    '--group',      'A',
    '--resume_csv', str(RESULTS_DIR / 'progress.csv'),
], check=False)   # check=False: idempotent; single failure must not abort

print(f'\nGroup A exit code: {_r.returncode}')
_n = len(list(RUNS_DIR.rglob('results_*.json')))
print(f'Completed so far: {_n}/170 runs')

# ── Drive backup ─────────────────────────────────────────────────────────────
print('\nBacking up Group A results to Drive ...')
DRIVE_JSON.mkdir(parents=True, exist_ok=True)
_backed = 0
for _src in sorted(RUNS_DIR.rglob('results_*.json')):
    _rel = _src.relative_to(RUNS_DIR)
    _dst = DRIVE_JSON / _rel
    _dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(_src, _dst)
    _backed += 1
print(f'Drive backup complete: {_backed} JSON files -> {DRIVE_JSON}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Group B: architecture & loss-weight sensitivity             ║
# ║                                                                         ║
# ║  Includes GLU-width ablations (meta_hid = 32, 64, 256) and LAUC       ║
# ║  weight sweep — 60 runs total (~40 unique GPU runs).                   ║
# ║  Estimated wall time on T4: ~7-8 h                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, shutil, os
from pathlib import Path

WORKING     = Path('/content')
PTB_ROOT    = WORKING / 'ptb-xl-1.0.3'
INDEX_PATH  = WORKING / 'index_complete.parquet'
RUNS_DIR    = WORKING / 'runs'
RESULTS_DIR = WORKING / 'results'
CODE_DIR    = Path(os.environ['EZNX_CODE_DIR'])
DRIVE_JSON  = Path('/content/drive/MyDrive/EZNX_ATLAS_A/json_results')
RESULTS_DIR.mkdir(exist_ok=True)

print('=' * 70)
print('GROUP B — Architecture & LAUC sensitivity (60 runs)')
print('=' * 70)

_r = subprocess.run([
    sys.executable, str(CODE_DIR / 'run_experiments_v2.py'),
    '--data_root',  str(PTB_ROOT),
    '--index_path', str(INDEX_PATH),
    '--runs_dir',   str(RUNS_DIR),
    '--group',      'B',
    '--resume_csv', str(RESULTS_DIR / 'progress.csv'),
], check=False)

print(f'\nGroup B exit code: {_r.returncode}')
_n = len(list(RUNS_DIR.rglob('results_*.json')))
print(f'Completed so far: {_n}/170 runs')

# ── Drive backup ─────────────────────────────────────────────────────────────
print('\nBacking up Group B results to Drive ...')
DRIVE_JSON.mkdir(parents=True, exist_ok=True)
_backed = 0
for _src in sorted(RUNS_DIR.rglob('results_*.json')):
    _rel = _src.relative_to(RUNS_DIR)
    _dst = DRIVE_JSON / _rel
    _dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(_src, _dst)
    _backed += 1
print(f'Drive backup complete: {_backed} JSON files -> {DRIVE_JSON}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Group C: 40 runs                                            ║
# ║  Estimated wall time on T4: ~7 h                                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, shutil, os
from pathlib import Path

WORKING     = Path('/content')
PTB_ROOT    = WORKING / 'ptb-xl-1.0.3'
INDEX_PATH  = WORKING / 'index_complete.parquet'
RUNS_DIR    = WORKING / 'runs'
RESULTS_DIR = WORKING / 'results'
CODE_DIR    = Path(os.environ['EZNX_CODE_DIR'])
DRIVE_JSON  = Path('/content/drive/MyDrive/EZNX_ATLAS_A/json_results')
RESULTS_DIR.mkdir(exist_ok=True)

print('=' * 70)
print('GROUP C — 40 runs')
print('=' * 70)

_r = subprocess.run([
    sys.executable, str(CODE_DIR / 'run_experiments_v2.py'),
    '--data_root',  str(PTB_ROOT),
    '--index_path', str(INDEX_PATH),
    '--runs_dir',   str(RUNS_DIR),
    '--group',      'C',
    '--resume_csv', str(RESULTS_DIR / 'progress.csv'),
], check=False)

print(f'\nGroup C exit code: {_r.returncode}')
_n = len(list(RUNS_DIR.rglob('results_*.json')))
print(f'Completed so far: {_n}/170 runs')

# ── Drive backup ─────────────────────────────────────────────────────────────
print('\nBacking up Group C results to Drive ...')
DRIVE_JSON.mkdir(parents=True, exist_ok=True)
_backed = 0
for _src in sorted(RUNS_DIR.rglob('results_*.json')):
    _rel = _src.relative_to(RUNS_DIR)
    _dst = DRIVE_JSON / _rel
    _dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(_src, _dst)
    _backed += 1
print(f'Drive backup complete: {_backed} JSON files -> {DRIVE_JSON}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Group D: 10 runs (~2 h)                                    ║
# ║  Can be run in the same session as Group C.                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, shutil, os
from pathlib import Path

WORKING     = Path('/content')
PTB_ROOT    = WORKING / 'ptb-xl-1.0.3'
INDEX_PATH  = WORKING / 'index_complete.parquet'
RUNS_DIR    = WORKING / 'runs'
RESULTS_DIR = WORKING / 'results'
CODE_DIR    = Path(os.environ['EZNX_CODE_DIR'])
DRIVE_JSON  = Path('/content/drive/MyDrive/EZNX_ATLAS_A/json_results')
RESULTS_DIR.mkdir(exist_ok=True)

print('=' * 70)
print('GROUP D — 10 runs')
print('=' * 70)

_r = subprocess.run([
    sys.executable, str(CODE_DIR / 'run_experiments_v2.py'),
    '--data_root',  str(PTB_ROOT),
    '--index_path', str(INDEX_PATH),
    '--runs_dir',   str(RUNS_DIR),
    '--group',      'D',
    '--resume_csv', str(RESULTS_DIR / 'progress.csv'),
], check=False)

print(f'\nGroup D exit code: {_r.returncode}')
_n = len(list(RUNS_DIR.rglob('results_*.json')))
print(f'Completed: {_n}/170 runs')
if _n == 170:
    print('ALL 170 RUNS COMPLETE. Proceed to Cell 13 (evaluation).')
else:
    print(f'{170 - _n} runs still pending — check RUNS_DIR and resume_csv.')

# ── Final Drive backup (all groups) ─────────────────────────────────────────
print('\nFinal Drive backup (all groups) ...')
DRIVE_JSON.mkdir(parents=True, exist_ok=True)
_backed = 0
for _src in sorted(RUNS_DIR.rglob('results_*.json')):
    _rel = _src.relative_to(RUNS_DIR)
    _dst = DRIVE_JSON / _rel
    _dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(_src, _dst)
    _backed += 1
print(f'Drive backup complete: {_backed} JSON files -> {DRIVE_JSON}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Post-training evaluation                                    ║
# ║  Run only AFTER all 170 training runs are complete.                    ║
# ║                                                                         ║
# ║  [13a] analyze_multiseed_v2   -> primary table, Wilcoxon, 95% CI      ║
# ║  [13b] compute_calibration    -> Brier + ECE + AUPRC                  ║
# ║  [13c] compute_subgroups      -> sex / age / anthropometric AUC        ║
# ║  [13d] evaluate_missingness   -> MCAR robustness table                ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, shutil, os
from pathlib import Path

WORKING     = Path('/content')
PTB_ROOT    = WORKING / 'ptb-xl-1.0.3'
INDEX_PATH  = WORKING / 'index_complete.parquet'
RUNS_DIR    = WORKING / 'runs'
RESULTS_DIR = WORKING / 'results'
CODE_DIR    = Path(os.environ['EZNX_CODE_DIR'])
DRIVE_RESULTS = Path('/content/drive/MyDrive/EZNX_ATLAS_A/analysis_results')

# Verify campaign is complete before running evaluation
_n_done = len(list(RUNS_DIR.rglob('results_*.json')))
print(f'Completed runs: {_n_done}/170')
if _n_done < 170:
    print(f'WARNING: {170 - _n_done} runs are missing.')
    print('Evaluation will use whichever runs are present (partial results may be less reliable).')


def _run_eval(label, cmd):
    print(f'\n=== {label} ===')
    _r = subprocess.run(cmd, capture_output=True, text=True)
    if _r.stdout:
        print(_r.stdout[-3000:])
    if _r.returncode != 0:
        print(f'STDERR ({label}):', _r.stderr[-1000:])
        print(f'WARNING: {label} exited with code {_r.returncode}')
    return _r.returncode


_run_eval('[13a] Statistical analysis (primary + sensitivity)', [
    sys.executable, str(CODE_DIR / 'analyze_multiseed_v2.py'),
    '--runs_dir', str(RUNS_DIR),
    '--out_dir',  str(RESULTS_DIR / 'stats'),
])

_run_eval('[13b] Calibration metrics (Brier + ECE + AUPRC)', [
    sys.executable, str(CODE_DIR / 'compute_calibration.py'),
    '--runs_dir', str(RUNS_DIR),
    '--out_dir',  str(RESULTS_DIR / 'calibration'),
])

_run_eval('[13c] Subgroup AUC (sex / age / anthropometric completeness)', [
    sys.executable, str(CODE_DIR / 'compute_subgroups.py'),
    '--runs_dir',   str(RUNS_DIR),
    '--index_path', str(INDEX_PATH),
    '--out_dir',    str(RESULTS_DIR / 'subgroups'),
])

_run_eval('[13d] Metadata missingness robustness (MCAR 0-100%)', [
    sys.executable, str(CODE_DIR / 'evaluate_missingness_v2.py'),
    '--runs_dir',   str(RUNS_DIR),
    '--index_path', str(INDEX_PATH),
    '--data_root',  str(PTB_ROOT),
    '--out_dir',    str(RESULTS_DIR / 'missingness'),
])

print('\n=== Evaluation complete. Output files: ===')
for _root, _dirs, _files in os.walk(str(RESULTS_DIR)):
    for _f in sorted(_files):
        _full = os.path.join(_root, _f)
        _rel  = os.path.relpath(_full, str(RESULTS_DIR))
        _sz   = os.path.getsize(_full)
        print(f'  {_rel:<65} {_sz:>10,} B')

# ── Back up analysis results to Drive ───────────────────────────────────────
print('\nBacking up analysis results to Drive ...')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
_backed = 0
for _src in sorted(RESULTS_DIR.rglob('*')):
    if _src.is_file():
        _rel = _src.relative_to(RESULTS_DIR)
        _dst = DRIVE_RESULTS / _rel
        _dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(_src, _dst)
        _backed += 1
print(f'Analysis results backed up: {_backed} files -> {DRIVE_RESULTS}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Package results and save to Drive                           ║
# ║                                                                         ║
# ║  JSON-only zip (~10 MB): suitable for all statistical analyses.        ║
# ║  Set INCLUDE_NPZ = True (~1.2 GB) to include raw probability arrays.  ║
# ║                                                                         ║
# ║  Output:                                                               ║
# ║    /content/results_package.zip   (local download via Colab sidebar)   ║
# ║    MyDrive/EZNX_ATLAS_A/results_package.zip  (persistent Drive copy)  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import zipfile, shutil, os
from pathlib import Path

# Set True only if you need raw probability arrays for downstream analysis
INCLUDE_NPZ = False

WORKING     = Path('/content')
RUNS_DIR    = WORKING / 'runs'
RESULTS_DIR = WORKING / 'results'
DRIVE_BASE  = Path('/content/drive/MyDrive/EZNX_ATLAS_A')
DRIVE_NPZ   = DRIVE_BASE / 'npz_archive'

zip_path = WORKING / 'results_package.zip'
_n_files = 0

print('Building results_package.zip ...')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as _zf:
    # results_*.json (one per run)
    for _p in sorted(RUNS_DIR.rglob('results_*.json')):
        _zf.write(_p, _p.relative_to(WORKING))
        _n_files += 1
    # analysis outputs (LaTeX tables, stats JSON, calibration, subgroups, missingness)
    for _p in sorted(RESULTS_DIR.rglob('*')):
        if _p.is_file():
            _zf.write(_p, _p.relative_to(WORKING))
            _n_files += 1
    # Optional: NPZ arrays
    if INCLUDE_NPZ:
        for _p in sorted(RUNS_DIR.rglob('probs_*.npz')):
            _zf.write(_p, _p.relative_to(WORKING))
            _n_files += 1

_mb = zip_path.stat().st_size / 1024**2
print(f'Package: {zip_path}  ({_n_files} files, {_mb:.1f} MB)')

# ── Copy zip to Drive ────────────────────────────────────────────────────────
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
_drive_zip = DRIVE_BASE / 'results_package.zip'
shutil.copy2(zip_path, _drive_zip)
print(f'\nDrive copy: {_drive_zip}  ({_drive_zip.stat().st_size / 1024**2:.1f} MB)')

# ── Optional: NPZ archive to Drive (large — only if INCLUDE_NPZ=True) ────────
if INCLUDE_NPZ:
    DRIVE_NPZ.mkdir(parents=True, exist_ok=True)
    _npz_zip = DRIVE_BASE / 'npz_archive.zip'
    print(f'\nBuilding NPZ archive -> {_npz_zip} ...')
    with zipfile.ZipFile(_npz_zip, 'w', compression=zipfile.ZIP_DEFLATED) as _zf:
        for _p in sorted(RUNS_DIR.rglob('probs_*.npz')):
            _zf.write(_p, _p.relative_to(WORKING))
    print(f'NPZ archive: {_npz_zip.stat().st_size / 1024**2:.0f} MB')

print('\n' + '=' * 70)
print('DONE. To download results_package.zip:')
print('  Colab sidebar (left panel) -> Files icon -> /content/results_package.zip')
print('  Or open Drive: drive.google.com -> EZNX_ATLAS_A/results_package.zip')
print('=' * 70)


## After downloading results_package.zip

The zip contains all JSON result files and LaTeX tables. The following scripts
produce all manuscript tables (no GPU required — runs on any laptop):

| Script | Output |
|--------|--------|
| `analyze_multiseed_v2.py` | Primary ablation table, Wilcoxon tests, 95% CI, sensitivity tables |
| `compute_calibration.py` | Brier score + ECE + AUPRC calibration table |
| `compute_subgroups.py` | Sex / age / anthropometric subgroup AUC table |
| `evaluate_missingness_v2.py` | MCAR missingness robustness table |

## Hardware provenance

- Platform: Google Colab — GPU: NVIDIA T4 (Colab Free tier)
- Seeds: 2024–2043, `CUBLAS_WORKSPACE_CONFIG=:4096:8`,
  `use_deterministic_algorithms(True)`, `cudnn.deterministic=True`,
  `cudnn.benchmark=False`, `allow_tf32=False` (matmul + cuDNN), `num_workers=0`

## Session time budget (Colab Free — T4, ~12 h per session)

| Session | Cells to run | Est. time |
|---------|-------------|----------|
| 1 (first) | 1→9 | ~11 h (incl. ~15 min PTB-XL download + ~10 h Group A) |
| 2 | 1→7 then 10 | ~8 h (incl. ~15 min download + ~7 h Group B) |
| 3 | 1→7 then 11→13 | ~10 h (Groups C+D ~9 h + evaluation ~1 h) |

> Cell 6 (Restore from Drive) ensures auto-resume across all sessions.
> The PTB-XL download (Cell 5) takes ~10-20 min per session since `/content/` is ephemeral.